# 2.0 — Construção do vector store (base de conhecimento)

Gera os embeddings das avaliações com o **Gemini** (`gemini-embedding-001`) e grava no **ChromaDB** em `data/vector_store/`.

Pré-requisitos:
- `pip install -r requirements.txt` na raiz do projeto
- `GEMINI_API_KEY` definida no ambiente ou no arquivo `.env` (veja `.env.example`)

A indexação é **retomável**: se parar por limite da API, basta executar de novo.

In [ ]:
import logging

import pandas as pd

from tech_challenge_fase4 import config
from tech_challenge_fase4.documentos import carregar_reviews, construir_documentos

logging.basicConfig(level=logging.INFO)
print(config.EMBEDDING_MODEL, config.EMBEDDING_DIM, config.COLLECTION_NAME, config.VECTOR_STORE_DIR)

## 1. Documentos

In [ ]:
df = carregar_reviews()
documentos = construir_documentos(df)
print(f"{len(df)} avaliações -> {len(documentos)} documentos")
documentos[0]

In [ ]:
tamanhos = pd.Series([len(d.texto) for d in documentos])
tamanhos.describe()

## 2. Indexação

Comece com uma amostra (`LIMITE = 500`) para validar a chave e o fluxo. Depois use `LIMITE = None` para indexar tudo.

In [ ]:
from tech_challenge_fase4.vector_store import construir_vector_store

LIMITE = 500  # None para indexar a base completa
colecao = construir_vector_store(limite=LIMITE)
colecao.count()

## 3. Busca

In [ ]:
from tech_challenge_fase4.retriever import buscar

pd.DataFrame(
    [
        {"similaridade": r["similaridade"], "nota": r["metadados"]["review_score"], "texto": r["texto"]}
        for r in buscar("produto chegou atrasado", k=5)
    ]
)

In [ ]:
# Com filtros por metadados
buscar("produto veio quebrado", k=5, filtros={"customer_state": "SP", "review_score": {"$lte": 2}})

## 4. Avaliação da recuperação

In [ ]:
from tech_challenge_fase4.avaliacao import avaliar

resultado = avaliar(k=10)
print(f"Precisão média: {resultado['precisao@10'].mean():.2%}")
resultado

## 5. Exemplo de RAG (referência para o time)

In [ ]:
from tech_challenge_fase4.retriever import responder

saida = responder("Quais são as principais reclamações sobre entrega?")
print(saida["resposta"])